# Loading the data and all imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split


from sklearn.linear_model import LinearRegression

In [ ]:
df = pd.read_csv("kalshi_w_macro_markets.csv")
df.head()

,Date,Close,High,Low,Open,Volume,Day Change %,3m_treasury,2yr_treasury,10yr_treasury,...,Overnight_Volatility,Futures_Last_Price,exp_rate,exp_rate_open_interest,volume_C25,volume_H0,volume_H25,mean_C25,mean_H0,mean_H25
0,2025-07-30,6362.899902,6396.540039,6336.379883,6381.229980,5375070000,-0.287250,4.41,3.94,4.38,...,0.000354,6412.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-07-31,6339.390137,6427.020020,6327.640137,6427.020020,6077080000,-1.363461,4.41,3.94,4.37,...,0.002083,6456.75,-1.000000,0.095081,30322.0,104894.0,93.0,-1.000000,0.115385,-1.0
2,2025-08-01,6238.009766,6287.279785,6212.689941,6287.279785,5827150000,-0.783646,4.35,3.69,4.23,...,0.002764,6314.75,-0.174537,-0.048415,749266.0,1152393.0,11073.0,0.820513,-0.672131,-1.0
3,2025-08-04,6329.939941,6330.689941,6271.709961,6271.709961,4842580000,0.928455,4.35,3.69,4.22,...,0.000658,6304.00,0.081891,-0.000429,72574.0,85485.0,7937.0,-0.027027,0.000000,-1.0
4,2025-08-05,6299.189941,6346.000000,6289.370117,6336.629883,5517410000,-0.590849,4.34,3.72,4.22,...,0.001891,6365.75,-0.652510,0.109144,169801.0,196020.0,108.0,0.027397,-0.105263,-1.0


In [ ]:
df = df.drop(columns=["mean_C25", "mean_H25", "exp_rate_open_interest"])
df = df.fillna(0)

KeyError: "['exp_rate_open_inerest'] not found in axis"

In [ ]:
target_col = "Day Change %"

X = df.drop(columns=["Date", target_col])
y = df[target_col]

print(X.shape)
print(y.shape)

(154, 23)
(154,)


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Baseline Linear Regression Model

In [40]:
linear_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression()),
    ]
)
linear_model.fit(X_train, y_train)

y_hat_train = linear_model.predict(X_train)
y_hat_test = linear_model.predict(X_test)

train_mse = mean_squared_error(y_train, y_hat_train)
test_mse = mean_squared_error(y_test, y_hat_test)

print("Linear Regression")
print("train_mse:", train_mse)
print("test_mse:", test_mse)

ValueError: could not convert string to float: '2026-01-16'

inspect the coefficients and the actual v. predicted 

In [7]:
coef_df = pd.DataFrame(
    {
        "feature": X.columns,
        "coefficient": linear_model.named_steps["regressor"].coef_,
    }
)
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
coef_df.head(10)


,feature,coefficient,abs_coefficient
0,Close,2.820090,2.820090
3,Open,-2.775969,2.775969
2,Low,-0.012776,0.012776
8,yield_curve,0.011366,0.011366
7,10yr_treasury,-0.010459,0.010459
6,2yr_treasury,0.009351,0.009351
1,High,0.007992,0.007992
10,fed_funds_rate,0.005000,0.005000
13,Futures_Last_Price,-0.002444,0.002444
5,3m_treasury,0.002430,0.002430


actual v predicted preview 

In [8]:
results_preview = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_hat_test
})

results_preview.head(10)

,Actual,Predicted
15,-0.169205,-0.159738
94,0.578610,0.590360
152,1.435721,1.436230
105,0.031000,0.031942
109,0.143058,0.145609
65,-0.556230,-0.573471
18,-0.284160,-0.274113
45,-0.237100,-0.235260
36,0.259511,0.251681
55,-0.896248,-0.882974


# Lasso Model

In [ ]:
from sklearn.linear_model import LassoCV

lasso_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LassoCV(cv=5, random_state=42)),
    ]
)
lasso_model.fit(X_train, y_train)

In [13]:
y_hat_train_lasso = lasso_model.predict(X_train)
y_hat_test_lasso = lasso_model.predict(X_test)

lasso_train_mse = mean_squared_error(y_train, y_hat_train_lasso)
lasso_test_mse = mean_squared_error(y_test, y_hat_test_lasso)

print("Lasso Regression")
print("train_mse:", lasso_train_mse)
print("test_mse:", lasso_test_mse)


Lasso Regression
train_mse: 0.0002476869252194533
test_mse: 0.00027048469866558565


In [14]:
lasso_coef_df = pd.DataFrame(
    {
        "feature": X.columns,
        "coefficient": lasso_model.named_steps["model"].coef_,
    }
)

lasso_coef_df["abs_coefficient"] = lasso_coef_df["coefficient"].abs()
lasso_coef_df = lasso_coef_df.sort_values("abs_coefficient", ascending=False)

lasso_coef_df.head(20)

,feature,coefficient,abs_coefficient
0,Close,2.804445,2.804445
3,Open,-2.765912,2.765912
13,Futures_Last_Price,-0.005828,0.005828
6,2yr_treasury,-0.002433,0.002433
10,fed_funds_rate,0.001671,0.001671
8,yield_curve,0.001598,0.001598
12,Overnight_Volatility,0.000770,0.000770
5,3m_treasury,0.000426,0.000426
4,Volume,-0.000036,0.000036
1,High,0.000000,0.000000


In [15]:
num_kept = (lasso_coef_df["coefficient"] != 0).sum()
num_dropped = (lasso_coef_df["coefficient"] == 0).sum()

print("Number of predictors kept by Lasso:", num_kept)
print("Number of predictors dropped by Lasso:", num_dropped)

Number of predictors kept by Lasso: 9
Number of predictors dropped by Lasso: 5


quick comparison of 2 models

In [17]:
comparison_df = pd.DataFrame(
    {
        "model": ["Linear Regression", "Lasso"],
        "train_mse": [train_mse, lasso_train_mse],
        "test_mse": [test_mse, lasso_test_mse],
    }   
)

comparison_df

,model,train_mse,test_mse
0,Linear Regression,0.000240,0.000313
1,Lasso,0.000248,0.000270


# Ridge Model

In [ ]:
from sklearn.linear_model import RidgeCV

ridge_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", RidgeCV(cv=5)),
    ]
)
ridge_model.fit(X_train, y_train)

In [19]:
y_hat_train_ridge = ridge_model.predict(X_train)
y_hat_test_ridge = ridge_model.predict(X_test)

ridge_train_mse = mean_squared_error(y_train, y_hat_train_ridge)
ridge_test_mse = mean_squared_error(y_test, y_hat_test_ridge)

print("Ridge Regression")
print("train_mse:", ridge_train_mse)
print("test_mse:", ridge_test_mse)

Ridge Regression
train_mse: 0.0010043231993296369
test_mse: 0.0009617937666737155


In [20]:
ridge_coef_df = pd.DataFrame(
    {
        "feature": X.columns,
        "coefficient": ridge_model.named_steps["model"].coef_,
    }
)
ridge_coef_df["abs_coefficient"] = ridge_coef_df["coefficient"].abs()
ridge_coef_df = ridge_coef_df.sort_values("abs_coefficient", ascending=False)
ridge_coef_df.head(20)

,feature,coefficient,abs_coefficient
0,Close,2.684192,2.684192
3,Open,-2.534333,2.534333
13,Futures_Last_Price,-0.171275,0.171275
2,Low,0.064622,0.064622
10,fed_funds_rate,-0.029200,0.029200
5,3m_treasury,0.026422,0.026422
6,2yr_treasury,-0.022792,0.022792
1,High,-0.019096,0.019096
9,VIX,-0.013413,0.013413
8,yield_curve,-0.010089,0.010089


comparison

In [21]:
comparison_df = pd.DataFrame(
    {
        "model": ["Linear Regression", "Lasso", "Ridge"],
        "train_mse": [train_mse, lasso_train_mse, ridge_train_mse],
        "test_mse": [test_mse, lasso_test_mse, ridge_test_mse],
    }
)
comparison_df

,model,train_mse,test_mse
0,Linear Regression,0.000240,0.000313
1,Lasso,0.000248,0.000270
2,Ridge,0.001004,0.000962


# Elastic Net Model

In [ ]:
from sklearn.linear_model import ElasticNetCV

elastic_net_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", ElasticNetCV(cv=5, random_state=42)),
    ]
)
elastic_net_model.fit(X_train, y_train)

In [23]:
y_hat_train_elastic = elastic_net_model.predict(X_train)
y_hat_test_elastic = elastic_net_model.predict(X_test)

elastic_train_mse = mean_squared_error(y_train, y_hat_train_elastic)
elastic_test_mse = mean_squared_error(y_test, y_hat_test_elastic)
print("Elastic Net Regression")
print("train_mse:", elastic_train_mse)
print("test_mse:", elastic_test_mse)

Elastic Net Regression
train_mse: 0.0002790320575781212
test_mse: 0.0002453625039131446


In [24]:
elastic_coef_df = pd.DataFrame(
    {
        "feature": X.columns,
        "coefficient": elastic_net_model.named_steps["model"].coef_,
    }
)
elastic_coef_df["abs_coefficient"] = elastic_coef_df["coefficient"].abs()
elastic_coef_df = elastic_coef_df.sort_values("abs_coefficient", ascending=False)
elastic_coef_df.head(20)

,feature,coefficient,abs_coefficient
0,Close,2.789788,2.789788
3,Open,-2.723060,2.723060
13,Futures_Last_Price,-0.036175,0.036175
6,2yr_treasury,-0.003684,0.003684
5,3m_treasury,0.003157,0.003157
2,Low,0.002456,0.002456
9,VIX,-0.001711,0.001711
8,yield_curve,0.001341,0.001341
12,Overnight_Volatility,0.000827,0.000827
11,Overnight_Return,-0.000737,0.000737


In [25]:
elastic_nonzero = elastic_coef_df[elastic_coef_df["coefficient"] != 0].copy()
elastic_nonzero.head(20)

,feature,coefficient,abs_coefficient
0,Close,2.789788,2.789788
3,Open,-2.723060,2.723060
13,Futures_Last_Price,-0.036175,0.036175
6,2yr_treasury,-0.003684,0.003684
5,3m_treasury,0.003157,0.003157
2,Low,0.002456,0.002456
9,VIX,-0.001711,0.001711
8,yield_curve,0.001341,0.001341
12,Overnight_Volatility,0.000827,0.000827
11,Overnight_Return,-0.000737,0.000737


compare 4 models so far 

In [26]:
comparison_df = pd.DataFrame(
    {
        "model": ["Linear Regression", "Lasso", "Ridge"],
        "train_mse": [train_mse, lasso_train_mse, ridge_train_mse],
        "test_mse": [test_mse, lasso_test_mse, ridge_test_mse],
    }
)
comparison_df

,model,train_mse,test_mse
0,Linear Regression,0.000240,0.000313
1,Lasso,0.000248,0.000270
2,Ridge,0.001004,0.000962


# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

random_forest_model = Pipeline(
    [
        ("model", RandomForestRegressor(n_estimators=100, random_state=42)),
    ]
)
random_forest_model.fit(X_train, y_train)   

In [28]:
y_hat_train_random_forest = random_forest_model.predict(X_train)
y_hat_test_random_forest = random_forest_model.predict(X_test)

random_forest_train_mse = mean_squared_error(y_train, y_hat_train_random_forest)
random_forest_test_mse = mean_squared_error(y_test, y_hat_test_random_forest)

print("Random Forest Regression")
print("train_mse:", random_forest_train_mse)
print("test_mse:", random_forest_test_mse)

Random Forest Regression
train_mse: 0.056857611212827654
test_mse: 0.3689649724202056


In [29]:
random_forest_importance_df = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": random_forest_model.named_steps["model"].feature_importances_,
    }
)
random_forest_importance_df = random_forest_importance_df.sort_values("importance", ascending=False)
random_forest_importance_df.head(20)

,feature,importance
9,VIX,0.248690
4,Volume,0.111077
0,Close,0.104854
12,Overnight_Volatility,0.098182
11,Overnight_Return,0.075710
3,Open,0.058054
8,yield_curve,0.053644
13,Futures_Last_Price,0.051043
7,10yr_treasury,0.050568
5,3m_treasury,0.044651


plot feauture imprtance

In [ ]:
import matplotlib.pyplot as plt

top_n = 15
plot_df = random_forest_importance_df.head(top_n).sort_values("importance")

plt.figure(figsize=(8, 6))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# Comparing all 5 models 

In [ ]:
comparison_df = pd.DataFrame(
    {
        "model": ["Linear Regression", "Lasso", "Ridge", "Elastic Net", "Random Forest"],
        "train_mse": [train_mse, lasso_train_mse, ridge_train_mse, elastic_train_mse, random_forest_train_mse],
        "test_mse": [test_mse, lasso_test_mse, ridge_test_mse, elastic_test_mse, random_forest_test_mse],
    }
)
comparison_df